[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/sequences_series_course.ipynb)

# Sequences and series full course: practice notebook

One practice question for every teaching part of the video, in the same order, and three cells you can move
yourself: a sequence built from any rule you type, an arithmetic progression with sliders for its first term and its
common difference, and the running totals of a geometric progression with a slider for its common ratio r. Work each
question out on paper first, then type your answer into the box and press **check**. It tells you whether you're
right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Just the numbers, without units. Whole numbers without commas: `10500` (a space is fine too:
`10 500`). Fractions as `5/11`, decimals as `102.4`. A rule for the nth term with `n`: `3n + 2`. You can type the
working too: `20/2*(18 + 75)`, `2^10`. For several answers, separate them with commas, in the order the question
asks. You can also put your answer between the quotes in a cell's last line, `practice(1, "")`, and run the cell: it
checks straight away.

**The cheat sheet.** Every formula from the course on two pages:
[sequences and series cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/sequences_series_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re, textwrap
from fractions import Fraction as F
from itertools import product
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD, VIOLET = "#6fb6bb", "#a0c27c", "#e8c56b", "#b49ad6"   # AP terms · GP terms · the answer · sums

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
n = sp.Symbol("n")
NAMES = {"n": n, "sqrt": sp.sqrt, "binomial": sp.binomial, "factorial": sp.factorial}
SUBS = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
def sub(k): return str(k).translate(SUBS)
UNITS = r"m\^2|\b(coins?|tiles?|seats?|tins?|blocks?|ways?|mm|cm|km|metres?|square|m)\b"   # (² is ^2 by now)

def prep(s):
    """Typed symbols → Python: − × ÷ · ² ³, 10 500 → 10500, units dropped."""
    s = s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("·", "*").replace("²", "^2").replace("³", "^3")
    s = re.sub(UNITS, "", s)
    return re.sub(r"(?<=\d) (?=\d{3}\b)", "", s).strip()

def read_one(p): return parse_expr(prep(p), local_dict=NAMES, transformations=RULES)

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [read_one(p) for p in s.split(",") if p.strip()]

def same(got, want, tol=1e-9):
    try:
        d = sp.simplify(sp.sympify(got) - sp.sympify(want))
        if d.free_symbols: return False                                 # a different rule in n
        return abs(float(sp.N(d, 30))) < tol
    except (TypeError, ValueError, AttributeError): return False

def dec(v, places=4):
    """A number as you'd write it: 3, 1.5, 0.6667, −12."""
    v = float(v)
    if abs(v - round(v)) < 1e-12: return f"{round(v)}".replace("-", "−")
    return f"{v:.{places}f}".rstrip("0").rstrip(".").replace("-", "−")

def frac(v):
    """An exact number: 3, 3/2, −12; a decimal when the fraction would be long."""
    v = sp.nsimplify(v)
    if v.is_Rational and v.q <= 99: return str(v).replace("-", "−")
    return dec(v)

# Each question: the question, the answers in order, the working, (for rounded answers) how close counts, and the move
# that cracks each part (shown when that part is wrong).
Q = {}

def practice(k, answer=""):
    """Question k: an answer box, a check button and a show-working button."""
    q = Q[k]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want, tol = q["answer"], q.get("tol", [1e-9] * len(q["answer"]))
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = [same(g, w, t) for g, w, t in zip(got, want, tol)]
            if all(ok): print("✓ right"); return
            for i, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {i}: right" if good else f"✗ part {i}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(textwrap.fill(line, 110))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"Question {k}. {q['text']}", 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def two_rows(N, terms, sums, title, dots=True, colour=SKY, line=None, limit=None, note="", note_col=DIM):
    """The terms on top (dots, or bars), and their running totals S₁, S₂, … as bars beneath them."""
    ns = np.arange(1, N + 1)
    fig, (top, bot) = plt.subplots(2, 1, figsize=(11.5, 5.8), sharex=True, gridspec_kw={"height_ratios": [1, 1]})
    fig.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.15, hspace=0.32)
    fig.text(0.08, 0.93, title, color=CHALK, fontsize=17)
    t = [float(v) for v in terms]; s = [float(v) for v in sums]
    if dots:
        if line is not None: top.plot(*line, color=colour, alpha=0.45, lw=1.6)
        top.scatter(ns, t, s=70, color=colour, zorder=3)
    else:
        top.bar(ns, t, width=0.62, color=colour)
    top.axhline(0, color=DIM, lw=1)
    top.set_title("the terms  a₁, a₂, a₃, …", loc="left", color=colour, fontsize=14)
    if N <= 12:                                                          # the values beside the dots
        span = (max(t + [0]) - min(t + [0])) or 1
        for k, v in zip(ns, terms):
            top.text(k, float(v) + 0.07 * span, frac(v), ha="center", va="bottom", color=CHALK, fontsize=11)
    bot.bar(ns, s, width=0.62, color=VIOLET)
    bot.axhline(0, color=DIM, lw=1)
    if limit is not None:
        bot.axhline(float(limit), color=GOLD, ls="--", lw=2)
        bot.text(N // 2 + 1, float(limit), "sum to infinity", color=GOLD, fontsize=12, ha="center", va="bottom")
    bot.set_title("the running totals  S₁, S₂, S₃, …", loc="left", color=VIOLET, fontsize=14)
    bot.set_xlabel("n")
    span = (max(s + [0]) - min(s + [0])) or 1
    lab = range(1, N + 1) if N <= 12 else list(range(1, 5)) + [N]
    for k in lab:
        v = s[k - 1]
        bot.text(k, v + (0.04 * span if v >= 0 else -0.04 * span), dec(sums[k - 1], 6 if k == N and N > 12 else 3), ha="center",
                 va="bottom" if v >= 0 else "top", color=CHALK, fontsize=10.5)
    for ax in (top, bot):
        lo, hi = ax.get_ylim(); pad = 0.18 * (hi - lo)
        ax.set_ylim(lo - (pad if lo < 0 else 0), hi + pad)
    top.set_xticks(ns if N <= 20 else ns[::2])
    if note: fig.text(0.08, 0.02, note, color=note_col, fontsize=14)
    plt.show()

## 1 · Sequences and series

A sequence is a list of numbers in order, like the heights of a bouncing ball: 1, 0.5, 0.25, 0.125, … metres. Each
number is a term, and a₁, a₂, a₃, …, aₙ name them by their place; aₙ is the nth term. A rule for the nth term takes
you straight to any term: aₙ = 2n + 1 gives 3, 5, 7, 9, …, and a₁₀₀ = 201. To ask whether a number is a term, set
the rule equal to it and solve: 3n − 2 = 100 gives n = 34, but 3n − 2 = 50 gives n = 52/3, which isn't a whole
number, so 50 isn't a term. A few terms don't fix the rule: points on a circle cut it into 1, 2, 4, 8, 16 pieces,
and then 31, not 32.

A recursive rule gives a place to start and a step: a₁ = 3, aₙ = aₙ₋₁ + 4 makes 3, 7, 11, 15, … The Fibonacci
sequence starts with two ones, and each new term is the sum of the two before it: 1, 1, 2, 3, 5, 8, 13, …

Add the terms up and you get a series. The running totals are S₁, S₂, S₃, …, where Sₙ is the sum of the first n
terms. Sigma, Σ, the Greek capital S, means "add up": the sum, from k = 1 to 5, of k² is 1 + 4 + 9 + 16 + 25 = 55.
Sums split, Σ(aₖ + bₖ) = Σaₖ + Σbₖ, and a number in front comes outside, Σ c aₖ = c Σ aₖ.

The next cell builds a sequence from any rule. Type the nth term in the box, using n (try `n^2`, `n/(n + 1)`,
`(-1)^n` or `3n - 2`), and press Enter. The terms appear on top, and their running totals grow beneath them. The
slider sets how many terms.

In [ ]:
def read_rule(rule):
    e = read_one(rule)
    if not e.free_symbols <= {n}: raise ValueError("only n, please")
    return e

def build(rule="2n + 1", terms=8):
    try:
        e = read_rule(rule)
        vals = [sp.nsimplify(e.subs(n, k)) for k in range(1, terms + 1)]
        if not all(v.is_real for v in vals): raise ValueError
    except Exception:
        print("Type the nth term using n, like 2n + 1, n^2 or n/(n + 1), then press Enter."); return
    sums = [sum(vals[:k]) for k in range(1, terms + 1)]
    shown = ", ".join(frac(v) for v in vals[:6]) + (", …" if terms > 6 else "")
    two_rows(terms, vals, sums, f"aₙ = {rule.strip()}:   {shown}", colour=CHALK,
             note=f"S{sub(terms)} = {' + '.join(frac(v) for v in vals[:4])}{' + …' if terms > 4 else ''} = {frac(sums[-1])}")

assert [read_rule("2n + 1").subs(n, k) for k in (1, 2, 3, 4, 100)] == [3, 5, 7, 9, 201]     # the video's rule
assert sum(k * k for k in range(1, 6)) == 55

In [ ]:
interact(build, rule=Text(value="2n + 1", description="aₙ =", continuous_update=False),
         terms=IntSlider(value=8, min=1, max=20, description="terms"));

In [ ]:
WAYS = {1: 1, 2: 2}                                                       # ways up the stairs, one or two steps at a time
for k in range(3, 9): WAYS[k] = WAYS[k - 1] + WAYS[k - 2]
def climbs(m): return sum(1 for j in range(m + 1) for s in product((1, 2), repeat=j) if sum(s) == m)
TILES = [3 * k + 2 for k in range(1, 11)]
Q[1] = {"text": "A garden path in Lisbon is laid in patterns of tiles: 5 tiles, then 8, then 11, then 14, and so on. "
                "Write a rule for the number of tiles in the nth pattern. Which pattern uses 101 tiles? And how many "
                "tiles do the first ten patterns use altogether: the sum, from k = 1 to 10, of (3k + 2)? Then, in "
                "Kyoto, you climb a staircase taking one or two steps at a time. In how many different ways can you "
                "climb 8 steps?",
        "answer": [3 * n + 2, [k for k in range(1, 100) if 3 * k + 2 == 101][0], sum(TILES), WAYS[8]],
        "moves": ["the patterns go up by 3, so the rule starts 3n. Then 3 × 1 + something = 5.",
                  "set 3n + 2 = 101 and solve for n.",
                  "split the sum: 3 × (1 + 2 + … + 10) + 2 × 10.",
                  "the last move is one step (from 7) or two steps (from 6), so the count is the sum of the two before "
                  "it: 1, 2, 3, 5, …"],
        "working": ["The patterns go up by 3 each time, so try 3n + something. For n = 1: 3 × 1 + 2 = 5, so aₙ = 3n + 2.",
                    "Check: a₄ = 3 × 4 + 2 = 14. ✓",
                    "3n + 2 = 101, so 3n = 99 and n = 33: the 33rd pattern uses 101 tiles.",
                    "The sum, from k = 1 to 10, of (3k + 2) = 3 × (1 + 2 + … + 10) + 2 × 10 = 3 × 55 + 20 = 185 tiles.",
                    "Stairs: 1 step can be climbed 1 way, 2 steps 2 ways (1 + 1 or 2).",
                    "Each count is the sum of the two before it: 1, 2, 3, 5, 8, 13, 21, 34. So 8 steps: 34 ways."]}
assert TILES[:4] == [5, 8, 11, 14] and Q[1]["answer"][1:] == [33, 185, 34]
assert sum(TILES) == 3 * 55 + 2 * 10 and WAYS[6] == 13                    # the video's sigma rules and its six stairs
assert all(climbs(k) == WAYS[k] for k in range(1, 9))                     # counted one by one
practice(1)

## 2 · Arithmetic progressions

An arithmetic progression (AP) goes up or down by the same amount each time: the common difference d, any term minus
the one before. A savings jar that starts with 50 coins and gets 20 more each week holds 50, 70, 90, 110, … The nth
term is aₙ = a + (n − 1)d: n terms have only n − 1 steps between them, like the gaps between 10 fence posts, so in
week 30 the jar holds 50 + 29 × 20 = 630 coins.

The sum of n terms is Sₙ = n/2 × (2a + (n − 1)d), or n/2 × (first + last). Write the sum forwards and backwards, and
every pair adds up to first + last: that's how 1 + 2 + … + 100 = 5050. The odd numbers add up to
squares: 1 + 3 + 5 + 7 = 16. A sum can have two answers: 24 + 21 + 18 + … reaches 78 after 4 terms and again after 13, because the
negative terms after 0 cancel the positive ones. The arithmetic mean of a and b is (a + b)/2, the middle of the two;
putting n numbers between two others makes n + 1 equal steps.

The next cell draws an AP. Move the sliders for the first term a and the common difference d: the terms line up on a
straight line, and the running total grows beneath them.

In [ ]:
def ap_terms(a, d, N): return [a + (k - 1) * d for k in range(1, N + 1)]
def ap_sum(a, d, N): return F(N, 2) * (2 * a + (N - 1) * d)

def ap_picture(a=3, d=4):
    N = 10
    t = ap_terms(a, d, N); s = [sum(t[:k]) for k in range(1, N + 1)]
    assert s[-1] == ap_sum(a, d, N) == F(N, 2) * (t[0] + t[-1])
    xs = np.linspace(0.6, N + 0.4, 50)
    two_rows(N, t, s, f"a = {a},  d = {d}:   aₙ = {a} + (n − 1) × {d}".replace("-", "−")
             .replace("+ (n − 1) × −", "− (n − 1) × "), colour=SKY, line=(xs, a + (xs - 1) * d),
             note=f"S₁₀ = 10/2 × (first + last) = 5 × ({frac(t[0])} {'−' if t[-1] < 0 else '+'} {frac(abs(t[-1]))}) = {frac(s[-1])}", note_col=GOLD)

In [ ]:
interact(ap_picture, a=IntSlider(value=3, min=-10, max=20, description="first term a"),
         d=IntSlider(value=4, min=-6, max=8, description="difference d"));

In [ ]:
SEATS = ap_terms(18, 3, 20)
Q[2] = {"text": "A cinema in Buenos Aires has 18 seats in its front row, and each row has 3 more seats than the row in "
                "front of it. There are 20 rows. How many seats are in the last row, and how many in the whole cinema? "
                "Then: four water stations stand between the 7 km mark and the 27 km mark of a race in Nairobi, so that "
                "all the gaps (from 7 to the first station, between stations, and from the last one to 27) are equal. "
                "How far apart are they?",
        "answer": [SEATS[-1], ap_sum(18, 3, 20), F(27 - 7, 4 + 1)],
        "moves": ["the nth term: a + (n − 1)d with a = 18, d = 3 and n = 20.",
                  "n/2 × (first + last), with the last row you just found.",
                  "four numbers in between make five equal steps."],
        "working": ["The rows make an AP with a = 18 and d = 3.",
                    "Last row: a₂₀ = 18 + (20 − 1) × 3 = 18 + 57 = 75 seats.",
                    "Whole cinema: S₂₀ = 20/2 × (18 + 75) = 10 × 93 = 930 seats.",
                    "(The other form agrees: 20/2 × (2 × 18 + 19 × 3) = 10 × 93 = 930.)",
                    "Stations: 7, _, _, _, _, 27 is an AP of six terms, so five equal steps.",
                    "The climb is 27 − 7 = 20 km, so each step is 20 ÷ 5 = 4 km: stations at 11, 15, 19 and 23 km."]}
assert Q[2]["answer"] == [75, 930, 4] and sum(SEATS) == 930 and ap_terms(7, 4, 6) == [7, 11, 15, 19, 23, 27]
assert ap_terms(50, 20, 30)[-1] == 630 and sum(range(1, 101)) == 5050                     # the video's jar and Gauss
assert [k for k in range(1, 30) if ap_sum(24, -3, k) == 78] == [4, 13]                    # the sum with two answers
practice(2)

## 3 · Geometric progressions

A geometric progression (GP) multiplies by the same number each time: the common ratio r, any term divided by the one
before. The ball's heights 1, 0.5, 0.25, … have r = 1/2. The nth term is aₙ = a rⁿ⁻¹, and growing by x percent means
multiplying by 1 + x/100: 1000 coins at 5% a year become 1000 × 1.05¹⁰ ≈ 1629 coins in 10 years.

The sum of n terms is Sₙ = a(1 − rⁿ)/(1 − r), and when r = 1 it's just n × a. Doubling grains on a chessboard gives
1 + 2 + 4 + … + 2⁶³ = 2⁶⁴ − 1. When r is between −1 and 1, rⁿ shrinks towards zero, and the running totals settle
towards the sum to infinity, a/(1 − r). The ball's heights add up to 1/(1 − 1/2) = 2, so with the 2 m drop it travels
2 + 2 × 2 = 6 metres. Recurring decimals are such sums: 0.2727… = 27/100 + 27/10000 + … = (27/100)/(1 − 1/100) =
27/99 = 3/11, and 0.999… = 1.

The next cell draws a GP's terms and its running totals. Move the slider for r. With r at a half, the totals settle
down towards two. Push r above one, and they run away. (The slider for a sets the first term.)

In [ ]:
def gp_terms(a, r, N): return [a * r ** (k - 1) for k in range(1, N + 1)]
def gp_sum(a, r, N): return F(a) * N if r == 1 else F(a) * (1 - F(r) ** N) / (1 - F(r))
def s_inf(a, r): return F(a) / (1 - F(r))

def partial_sums(r=0.5, a=1.0):
    N = 20
    r, a = F(round(r * 100), 100), F(round(a * 10), 10)                 # the slider's value, exactly
    t = gp_terms(a, r, N); s = [sum(t[:k]) for k in range(1, N + 1)]
    assert s[-1] == gp_sum(a, r, N)
    R, A = dec(r, 2), dec(a, 1)
    if -1 < r < 1:
        lim = s_inf(a, r)
        value = f"= {frac(lim)}" if lim.denominator <= 99 else f"≈ {dec(lim)}"
        note, col = f"r = {R} is between −1 and 1, so the totals settle towards a/(1 − r) = {A}/(1 {'+' if r < 0 else '−'} {dec(abs(r), 2)}) {value}", GOLD
    else:
        lim = None
        if r == 1: note = f"r = 1: every term is {A}, so Sₙ = n × {A}, and the totals run away"
        elif r > 1: note = f"r = {R} is above 1: each term is bigger than the one before, so the totals run away"
        elif r == -1: note = f"r = −1: the totals swing between {A} and 0 and never settle"
        else: note = f"r = {R} is below −1: the terms swing and grow, so the totals never settle"
        col = CORAL
    totals = ",  ".join(f"S{sub(k)} = {dec(s[k - 1])}" for k in range(1, 5))
    two_rows(N, t, s, f"a = {A},  r = {R}:   {totals},  …,  S₂₀ = {dec(s[-1], 6)}", dots=False, colour=MINT,
             limit=lim, note=note, note_col=col)

assert s_inf(1, F(1, 2)) == 2 and [sum(gp_terms(F(1), F(1, 2), k)) for k in (1, 2, 3, 4)] == [1, F(3, 2), F(7, 4), F(15, 8)]
assert 2 + 2 * s_inf(1, F(1, 2)) == 6 and s_inf(F(27, 100), F(1, 100)) == F(3, 11)        # the ball; 0.2727…
assert gp_sum(1, 2, 64) == 2 ** 64 - 1 and round(1000 * 1.05 ** 10) == 1629
assert gp_sum(1, F(11, 10), 20) > 50                                     # r above one: the totals run away

In [ ]:
interact(partial_sums, r=FloatSlider(value=0.5, min=-1.5, max=1.5, step=0.05, readout_format=".2f", description="ratio r"),
         a=FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5, readout_format=".1f", description="first term a"));

In [ ]:
FOLDS = gp_terms(F(2, 10), 2, 10)                                         # thickness (mm) after 1, 2, …, 10 folds
Q[3] = {"text": "A sheet of paper 0.1 mm thick is folded in half again and again. How thick is the stack, in mm, after "
                "10 folds? A savings club in Accra asks you to put in 1 coin on the first day, 2 on the second, 4 on "
                "the third, doubling every day. How many coins have you put in altogether after 12 days? And a "
                "calculator shows 0.454545… (the 45 repeating for ever). Write it as a fraction.",
        "answer": [FOLDS[-1], gp_sum(1, 2, 12), s_inf(F(45, 100), F(1, 100))],
        "moves": ["each fold doubles the thickness: 0.1 × 2 × 2 × … ten times.",
                  "a = 1, r = 2, n = 12 in Sₙ = a(rⁿ − 1)/(r − 1).",
                  "0.4545… = 45/100 + 45/10000 + …: a = 45/100 and r = 1/100, so the sum is a/(1 − r)."],
        "working": ["After 1 fold the stack is 0.2 mm, then 0.4, 0.8, …: a GP with a = 0.2 and r = 2.",
                    "a₁₀ = 0.2 × 2⁹ = 0.2 × 512 = 102.4 mm, just over 10 cm. (That's 0.1 × 2¹⁰: 1024 layers.)",
                    "Coins: 1 + 2 + 4 + … + 2¹¹ is a GP with a = 1, r = 2 and n = 12.",
                    "S₁₂ = 1 × (2¹² − 1)/(2 − 1) = 4096 − 1 = 4095 coins.",
                    "0.4545… = 45/100 + 45/10000 + 45/1000000 + …, with a = 45/100 and r = 1/100.",
                    "r is between −1 and 1, so the sum is (45/100)/(1 − 1/100) = (45/100)/(99/100) = 45/99 = 5/11.",
                    "Check: 5 ÷ 11 = 0.454545…"]}
assert Q[3]["answer"] == [F(512, 5), 4095, F(5, 11)] and F(1, 10) * 2 ** 10 == F(512, 5) and 2 ** 10 == 1024
assert str(5 / 11)[:8] == "0.454545" and sum(2 ** k for k in range(12)) == 4095
practice(3)

## 4 · The arithmetic and geometric means

The geometric mean of two positive numbers a and b is G = √(ab): the middle term that makes a, G, b a GP. Between 4
and 9 it's 6, because 4, 6, 9 multiply by 3/2 each time. Putting n numbers between two others makes a GP of n + 2
terms, so the ratio comes from the last term: between 3 and 81, two numbers make 3 × r³ = 81, r = 3, and the numbers
are 9 and 27.

The arithmetic mean A = (a + b)/2 is never smaller than G: A − G = (√a − √b)²/2, a square divided by two, so it's
never negative, and A = G only when a = b. In a semicircle on a diameter a + b, the radius is A and the half-chord
at the join is G. So of all rectangles with the same perimeter, the square has the largest area: sides x and
20 − x give at most 10 × 10 = 100. And if two numbers have A = 10 and G = 8, they add up to 20 and multiply to 64:
they're 16 and 4.

In [ ]:
def gm(a, b): return sp.sqrt(sp.Integer(a) * b)
FENCE = max(F(x, 2) * (24 - F(x, 2)) for x in range(0, 49))              # sides x and 24 − x, in half-metres
A4, G4 = 13, 12                                                           # the two numbers: they add to 2A and multiply to G²
PAIR = sorted(sp.solve(sp.Symbol("t") ** 2 - 2 * A4 * sp.Symbol("t") + G4 ** 2), reverse=True)
Q[4] = {"text": "Put two numbers between 5 and 135 so that the four numbers make a GP: what are the two numbers, smaller "
                "first? A farmer in Chile has 48 m of fencing for a rectangular pen. What's the largest area it can "
                "enclose, in square metres? And two numbers have an arithmetic mean of 13 and a geometric mean of 12. "
                "What are they, larger first?",
        "answer": [5 * 3, 5 * 9, FENCE, PAIR[0], PAIR[1]],
        "moves": ["four terms: 5 × r³ = 135, so r³ = 27.", "the same: then multiply by r again.",
                  "the half-perimeter is 24 m. Of all rectangles with the same perimeter, the square has the largest area.",
                  "they add up to 2 × 13 = 26 and multiply to 12² = 144.", "26 minus the larger one."],
        "working": ["5, _, _, 135 is a GP of four terms: 5 × r³ = 135, so r³ = 27 and r = 3.",
                    "The two numbers are 5 × 3 = 15 and 15 × 3 = 45 (and 45 × 3 = 135 ✓).",
                    "The sides are x and 24 − x. Their arithmetic mean is 12, and the area is the geometric mean squared.",
                    "A ≥ G, so the area is at most 12² = 144 m², reached when x = 24 − x = 12: a 12 m by 12 m square.",
                    "The numbers add up to 2 × 13 = 26 and multiply to 12² = 144.",
                    "(a − b)² = (a + b)² − 4ab = 676 − 576 = 100, so a − b = 10. Then a = 18 and b = 8.",
                    "Check: (18 + 8)/2 = 13 and √(18 × 8) = √144 = 12. ✓"]}
assert Q[4]["answer"] == [15, 45, 144, 18, 8] and 5 * 3 ** 3 == 135 and gm(18, 8) == 12
assert gm(4, 9) == 6 and 3 * 3 ** 3 == 81 and max(x * (20 - x) for x in range(21)) == 100   # the video's means
assert all(F(a + b, 2) >= gm(a, b) for a in range(1, 30) for b in range(1, 30))            # A ≥ G
practice(4)

## 5 · Sequences in money and growth

Simple interest pays the same amount every year, so the balance is an AP: 1000 coins at 6% simple interest grow by
60 a year, to 1600 after 10 years. Compound interest multiplies by the same number every year, so the balance is a
GP: 1000 × 1.06¹⁰ ≈ 1791 coins after 10 years. The gap widens: after 30 years, 2800 against about 5743.

Saving the same amount at the end of every year adds up a GP: 1000 coins a year for 5 years at 5% gives 1000 ×
(1.05⁵ − 1)/0.05 ≈ 5526 coins, of which about 526 is interest. A loan paid back in equal payments works the same way
backwards: 33 100 coins at 10% a year, paid back in three equal yearly payments, needs 33 100 × 1.1³ / (1.1² + 1.1 +
1) = 13 310 coins a year. A town of 50 000 growing 2% a year passes 60 000 in its tenth year.

Amounts are in coins, rounded to the nearest coin at the end.

In [ ]:
def coins(x): return int(F(x) + F(1, 2))                                   # to the nearest coin (x ≥ 0)
GROW = 2000 * F(105, 100) ** 8
SAVED = sum(500 * F(104, 100) ** k for k in range(4))                    # deposits at the ends of years 1–4, at year 4
PAY = 10500 * F(11, 10) ** 2 / (F(11, 10) + 1)
Q[5] = {"text": "A shop owner in Cape Town puts 2000 coins in an account paying 5% a year compound interest. How much is "
                "there after 8 years, to the nearest coin? Her neighbour saves 500 coins at the end of every year for 4 "
                "years, in an account paying 4% a year. How much is in it just after the fourth deposit, to the "
                "nearest coin? And a family in Istanbul borrows 10 500 coins at 10% a year, and pays it back in two "
                "equal payments, one at the end of each year. How much is each payment?",
        "answer": [coins(GROW), coins(SAVED), PAY], "tol": [0.51, 0.51, 1e-9],
        "moves": ["compound interest is a GP: multiply by 1.05 eight times.",
                  "the first deposit grows for 3 years, the last for none: 500 × (1 + 1.04 + 1.04² + 1.04³).",
                  "after a year the debt is 10 500 × 1.1 − P; a year later that times 1.1 must equal P."],
        "working": ["2000 × 1.05⁸ = 2000 × 1.4774554… = 2954.91…, so about 2955 coins.",
                    "The four deposits, grown to the end of year 4: 500 × 1.04³ + 500 × 1.04² + 500 × 1.04 + 500.",
                    "That's a GP: 500 × (1.04⁴ − 1)/(1.04 − 1) = 500 × 4.246464 = 2123.232, so about 2123 coins.",
                    "Loan: after one year the family owes 10 500 × 1.1 − P = 11 550 − P.",
                    "A year later that grows by 10% and the second payment clears it: (11 550 − P) × 1.1 = P.",
                    "So 12 705 = 2.1 P, and P = 12 705 ÷ 2.1 = 6050 coins.",
                    "Check: 10 500 + 1050 − 6050 = 5500, then 5500 + 550 − 6050 = 0. ✓"]}
assert Q[5]["answer"] == [2955, 2123, 6050] and round(float(GROW), 2) == 2954.91 and float(SAVED) == 2123.232
assert (10500 + 1050 - PAY) * F(11, 10) == PAY
assert coins(1000 * F(106, 100) ** 10) == 1791 and coins(1000 * (F(105, 100) ** 5 - 1) / F(5, 100)) == 5526   # the video
assert 33100 * F(11, 10) ** 3 / (F(11, 10) ** 2 + F(11, 10) + 1) == 13310
practice(5)

## 6 · Sums of special series

Three sums come up again and again. The first n whole numbers: 1 + 2 + … + n = n(n + 1)/2, so 1 to 20 makes 210.
The squares: 1² + 2² + … + n² = n(n + 1)(2n + 1)/6. Three copies of a pyramid of squares fit into a rectangle (2n + 1)
by n(n + 1)/2, and 1 + 4 + 9 + … + 100 = 385. The cubes: 1³ + 2³ + … + n³ = [n(n + 1)/2]², the square of the
first sum, so the cubes up to 10³ add up to 55² = 3025.

A series built from these sums comes apart into them: 1 × 2 + 2 × 3 + … + n(n + 1) = Σk² + Σk = n(n + 1)(n + 2)/3.
For a sum that starts in the middle, take two of them away: 5² + 6² + … + 20² = (1² + … + 20²) − (1² + … + 4²) =
2870 − 30 = 2840.

In [ ]:
def S1(m): return m * (m + 1) // 2
def S2(m): return m * (m + 1) * (2 * m + 1) // 6
def S3(m): return S1(m) ** 2
Q[6] = {"text": "A shop in Mexico City stacks tins in a square pyramid: 1 tin on top, then 4, then 9, and so on, with 8 "
                "layers. How many tins are there? A child builds solid cubes from toy blocks, with edges of 1, 2, 3, "
                "…, 8 blocks. How many blocks does she use altogether? And add up 1 × 2 + 2 × 3 + 3 × 4 + … + 10 × 11.",
        "answer": [S2(8), S3(8), sum(k * (k + 1) for k in range(1, 11))],
        "moves": ["1² + 2² + … + 8²: use n(n + 1)(2n + 1)/6 with n = 8.",
                  "1³ + 2³ + … + 8³ is the square of 1 + 2 + … + 8.",
                  "k(k + 1) = k² + k, so add the sum of squares and the sum of whole numbers, up to 10."],
        "working": ["Tins: 1² + 2² + … + 8² = 8 × 9 × 17 / 6 = 1224 / 6 = 204 tins.",
                    "Blocks: 1³ + 2³ + … + 8³ = (8 × 9 / 2)² = 36² = 1296 blocks.",
                    "1 × 2 + 2 × 3 + … + 10 × 11 = (1² + … + 10²) + (1 + … + 10) = 385 + 55 = 440.",
                    "The formula agrees: n(n + 1)(n + 2)/3 = 10 × 11 × 12 / 3 = 440."]}
assert Q[6]["answer"] == [204, 1296, 440] == [sum(k * k for k in range(1, 9)), sum(k ** 3 for k in range(1, 9)),
                                              10 * 11 * 12 // 3]
assert S1(20) == 210 and S2(10) == 385 and S3(10) == 3025 and S2(20) - S2(4) == 2840        # the video's sums
practice(6)

## Sources
NCERT Mathematics Class XI, ch. 8 (Sequences and Series); OpenStax, Precalculus 2e, ch. 11 (Sequences, Probability
and Counting Theory). Both retrieved 2026-11-06. Every question is our own.